In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
### Load the trained model,onehotencoder, scalar pickle 
model = load_model('model.h5')

## load the encoder and scalar
with open('one_hot_encoder.pkl','rb') as file:
    one_hot_encoder = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
     label_encoder_gender= pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)



In [3]:
input_data ={
    	'CreditScore' : 450,
          'Geography' : 'Germany', 
          'Gender' : 'Female' ,
          'Age' : 55,
        	'Tenure': 1 ,
        	'Balance' : 150000,
            'NumOfProducts': 1,
            'HasCrCard':1,
        	'IsActiveMember' : 0,
        	'EstimatedSalary' : 30000	}

In [4]:
hot_encoded= one_hot_encoder.transform([[input_data['Geography']]]).toarray()
hot_encoded_df= pd.DataFrame(hot_encoded, columns=one_hot_encoder.get_feature_names_out(['Geography']))
hot_encoded_df

c:\Users\iitia\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,0.0,1.0,0.0


In [5]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,450,Germany,Female,55,1,150000,1,1,0,30000


In [6]:
## Encode categorical variables
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,450,Germany,0,55,1,150000,1,1,0,30000


In [7]:
## concatinate one hot encode
input_df = input_df.drop("Geography", axis=1)
input_df = pd.concat([input_df,hot_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,450,0,55,1,150000,1,1,0,30000,0.0,1.0,0.0


In [8]:
##Scalling the data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-2.09264482, -1.09499335,  1.53088014, -1.38944225,  1.18317786,
        -0.91668767,  0.64920267, -1.02583358, -1.22456561, -0.99850112,
         1.72572313, -0.57638802]])

In [9]:
## Prediction
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step


array([[0.9428936]], dtype=float32)

In [10]:
prediction_proba = prediction[0][0]


In [11]:
prediction_proba

np.float32(0.9428936)

In [12]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

The customer is likely to churn.
